## Opensearch 체크

In [2]:
from opensearchpy import OpenSearch
from utils.rag_utils_open import get_opensearch_client
import json, yaml
with open('config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# OpenSearch 클라이언트 생성
client = get_opensearch_client()

# 확인할 인덱스 이름 정의
SUSI_INDEX = "susi_index"
KGS_INDEX = "kgs_index"

print('인덱스 종류')
print(client.cat.indices(v=True))

인덱스 종류
health status index                                                  uuid                   pri rep docs.count docs.deleted store.size pri.store.size
green  open   .plugins-ml-model-group                                2eypuKpWSBy9sJKeiKaKpg   1   0          1            0      5.9kb          5.9kb
yellow open   kgs_index_opensearch_k1_1_2_b_0_75_m_14_efs_64_efc_128 Zv5BHyHESzWWIxOLZS5iFQ   5   1        203            0     14.2mb         14.2mb
green  open   .plugins-flow-framework-state                          M_yjgcJiRD6cNpFuonFoDA   5   0          2            0      8.1kb          8.1kb
green  open   .ql-datasources                                        O7UjUW1rTTKFvWKViOVDfA   1   0          0            0       208b           208b
green  open   .plugins-ml-agent                                      Gt7NJuzdTfWctsWm4lQzhw   1   0          1            0     16.6kb         16.6kb
green  open   .plugins-ml-task                                       GOADYf38QhKLZ_kPt6pnGA  

### 인덱스 내용 조회

In [ ]:
# 스키마 조회
target_index = KGS_INDEX
response = client.indices.get(index=target_index) # 인덱스 정보 조회 (설정 + 매핑)

print(json.dumps(response, indent=2, ensure_ascii=False))

{
  "kgs_index_opensearch_k1_1_2_b_0_75_m_32_efs_64_efc_128": {
    "aliases": {},
    "mappings": {
      "properties": {
        "metadata": {
          "properties": {
            "category1": {
              "type": "keyword"
            },
            "category2": {
              "type": "keyword"
            },
            "category3": {
              "type": "keyword"
            },
            "question": {
              "type": "text",
              "analyzer": "standard"
            },
            "question_morph": {
              "type": "text",
              "analyzer": "korean_nori"
            },
            "text_morph": {
              "type": "text",
              "analyzer": "korean_nori"
            },
            "university": {
              "type": "keyword"
            },
            "year": {
              "type": "integer"
            }
          }
        },
        "text": {
          "type": "text",
          "analyzer": "standard"
        },
        "vector

In [3]:
def query_index(index_name, size):
    if client.indices.exists(index=index_name):
        response = client.search(
            index=index_name,
            body={
                "query": {
                    "match_all": {}
                },
                "size": size
            }
        )
    
    return response

response= query_index('kgs_index_opensearch_k1_1_2_b_0_75_m_16_efs_64_efc_128', 5)
response

{'took': 2,
 'timed_out': False,
 '_shards': {'total': 5, 'successful': 5, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 1019, 'relation': 'eq'},
  'max_score': 1.0,
  'hits': [{'_index': 'kgs_index_opensearch_k1_1_2_b_0_75_m_16_efs_64_efc_128',
    '_id': 'zlX-uJoBEshXRSnsD4S-',
    '_score': 1.0,
    '_source': {'text': '✅ 3년 특례\n지원자격: 해외에서 중·고교 과정 중 합산 3년 이상 재학, 그중 반드시 고교 1년 이상 포함해야 함.\n보호자 요건: 부모 중 1인이 해외에서 합법적으로 취업·사업·주재원 등의 자격으로 3년 이상 체류해야 함.\n주요 대상: 주재원, 파견 근무, 교포 자녀 등 중·장기 체류자.\n\n✅ 12년 특례\n지원자격: 초·중·고 전 교육과정을 해외에서 12년 전부 이수해야 함.\n보호자 요건: 부모 체류 조건은 제한 없음\n주요 대상: 대한민국 국적을 가진 해외 전교육과정 이수자',
     'metadata': {'year': 2026,
      'university': '공통',
      'category1': '전형이해',
      'category2': '3년, 12년특례',
      'category3': '3년, 12년특례 비교',
      'question': '재외국민특별전형 3년, 12년특례는 어떻게 구분되나요?',
      'question_morph': '재외국민특별전형 3년, 12년특례는 어떻게 구분되나요?',
      'text_morph': '✅ 3년 특례\n지원자격: 해외에서 중·고교 과정 중 합산 3년 이상 재학, 그중 반드시 고교 1년 이상 포함해야 함.\n보호자 요건: 부모 중 1인이 해외에서 합법적으로 취업·사업·

### 인덱스 삭제

In [9]:
def delete_index(client, index_name):
    try:
        # 인덱스가 존재하는지 먼저 확인
        if client.indices.exists(index=index_name):
            client.indices.delete(index=index_name)
            print(f"✅ 인덱스 '{index_name}'를 성공적으로 삭제했습니다.")
            return True
        else:
            print(f"⚠️ 인덱스 '{index_name}'가 존재하지 않습니다.")
            return False
    except Exception as e:
        print(f"❌ 인덱스 삭제 중 오류가 발생했습니다: {e}")
        return False

delete_index(client, "kgs_index")

✅ 인덱스 'kgs_index'를 성공적으로 삭제했습니다.


True

In [2]:
# opensearch k1, b index 삭제 
from utils.rag_utils_open import get_opensearch_client

# OpenSearch 클라이언트 생성
client = get_opensearch_client()
indices_info = client.cat.indices(format="json")
index_list = [idx_info['index'] for idx_info in indices_info]

print("--- 전체 인덱스 리스트 ---")
print(index_list)

# (옵션) .kibana 같은 시스템 인덱스를 제외하고 싶을 경우
user_index_list = [
    idx_info['index'] for idx_info in indices_info 
    if not idx_info['index'].startswith('.')
]

print("\n--- 시스템 인덱스 제외 리스트 ---")
print(user_index_list)

user_index_list.remove('kgs_index')
user_index_list.remove('susi_index')

def delete_index(client, index_name):
    try:
        # 인덱스가 존재하는지 먼저 확인
        if client.indices.exists(index=index_name):
            client.indices.delete(index=index_name)
            print(f"✅ 인덱스 '{index_name}'를 성공적으로 삭제했습니다.")
            return True
        else:
            print(f"⚠️ 인덱스 '{index_name}'가 존재하지 않습니다.")
            return False
    except Exception as e:
        print(f"❌ 인덱스 삭제 중 오류가 발생했습니다: {e}")
        return False

for i in user_index_list:
    delete_index(client, i)

--- 전체 인덱스 리스트 ---
['.plugins-ml-model-group', '.plugins-flow-framework-state', '.ql-datasources', '.plugins-ml-agent', '.plugins-ml-task', '.plugins-flow-framework-templates', 'kgs_index_opensearch_k1_1_2_b_0_75_m_16_efs_64_efc_128', '.kibana_1', '.opendistro_security', '.opensearch-observability', '.plugins-ml-config', '.plugins-ml-model', '.plugins-flow-framework-config', 'kgs_index_opensearch_vec_m16_efc128', 'susi_index', 'kgs_index']

--- 시스템 인덱스 제외 리스트 ---
['kgs_index_opensearch_k1_1_2_b_0_75_m_16_efs_64_efc_128', 'kgs_index_opensearch_vec_m16_efc128', 'susi_index', 'kgs_index']
✅ 인덱스 'kgs_index_opensearch_k1_1_2_b_0_75_m_16_efs_64_efc_128'를 성공적으로 삭제했습니다.
✅ 인덱스 'kgs_index_opensearch_vec_m16_efc128'를 성공적으로 삭제했습니다.
